# 부산항 컨테이너 성장에서 환적의 기여 구조 파악

- 연간 총증감량에서 환적과 수출입이 각각 기여한 양 분석
- 총물동량 증가와 환점점유율의 장기 변화 이해
- 전년 대비 증감량을 환적과 수출입으로 분해하여 성장 기여 해석

# 데이터 메타데이터
- 원본 파일명: 부산광역시_부산항 물동량 현황_20251232.csv
- 전체 행: 13
- 출처: [공공데이터 포털](https://www.data.go.kr/data/3076569/fileData.do)
- 비고:
  - 단위: 천 TEU (1,000 TEU)
  - 상세 사항 참고처
    - BPA 부산항만공사 체인 포털
    - 해양수산부 PORT-MIS

In [1]:
import pandas as pd
import numpy as np



In [13]:
pd.set_option("display.float_format", lambda value: format(value, '.3f'))

In [14]:
FILE_NAME = 'transshipment_data.csv'
DATA_FILE_PATH = f'../data/{FILE_NAME}'

In [15]:
origin = pd.read_csv(
    DATA_FILE_PATH,
    encoding='euc-kr', # cp949가 아니더라도 괜찮습니다. 읽어올 수만 있으면 됩니다.
)

FileNotFoundError: [Errno 2] No such file or directory: '../data/transshipment_data.csv'

# 데이터확인

In [12]:
origin.head()

NameError: name 'origin' is not defined

# 데이터 형태 확인

In [6]:
origin.shape

(13, 9)

In [7]:
origin.columns.to_list()

['연도', ' 컨물동량', '증감율', '수출입소계', '수입', '수출', '환적소계', '환적점유율', '환적증감율']

확인하여보니, `컨물동량` 열(Column)에서 앞에 공백이 포함된 것을 확인할 수 있습니다. 이렇게 되면 이후에 분석할 때 방해가될 수 있으니 다음과 같이 공백을 제거하여 적용하도록 하겠습니다.

In [8]:
origin.columns = origin.columns.str.strip()

In [9]:
origin.columns.to_list()

['연도', '컨물동량', '증감율', '수출입소계', '수입', '수출', '환적소계', '환적점유율', '환적증감율']

In [10]:
origin.dtypes

연도         int64
컨물동량       int64
증감율      float64
수출입소계      int64
수입         int64
수출         int64
환적소계       int64
환적점유율    float64
환적증감율    float64
dtype: object

In [11]:
origin.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   연도      13 non-null     int64  
 1   컨물동량    13 non-null     int64  
 2   증감율     13 non-null     float64
 3   수출입소계   13 non-null     int64  
 4   수입      13 non-null     int64  
 5   수출      13 non-null     int64  
 6   환적소계    13 non-null     int64  
 7   환적점유율   13 non-null     float64
 8   환적증감율   13 non-null     float64
dtypes: float64(3), int64(6)
memory usage: 1.0 KB


# 결측치 확인

In [12]:
origin.isna().sum()

연도       0
컨물동량     0
증감율      0
수출입소계    0
수입       0
수출       0
환적소계     0
환적점유율    0
환적증감율    0
dtype: int64

지금까지의 내용의 내용을 종합하면 모든 열이 숫자형이지만 연도는 식별자, 증감율과 환적점유율은 백분율, 나머지는 천 TEU 수준의 물동량이어서 단위가 다르게 되어있습니다.

총 13개 관측치만으로 평균과 표준편차를 계산할 수는 있지만 소수 연도의 충격에 민감하다는 판단입니다. 2025년이 완전 연간 실적인지 전망 또는 잠정치인지는 info()로 알 수 없으므로 자료 설명과 기준일을 별도로 확인해야 합니다.

공공데이터 포털을 확인하면 업데이트 주기는 연간으로 되어있고 등록일은 2025년 3월 24일이나, 수정일이 2026년 4월 14일인 것으로 미루어보아, 2025년이 완전 연간 실적일 가능성이 높습니다.

또한, 이 데이터에는 연도, 컨물동량, 수출입소계, 수입, 수출, 환적소계, 환적점유율, 증감율, 환적증감율이 있습니다. 즉, 단순히 전체 컨테이너 물동량 하나만 있는 것이 아니라, 전체 물동량과 그 안의 수출입, 환적 구조, 그리고 연도라는 시간 정보가 함께 존재하고 있는 것입니다.

따라서 데이터를 보는 순간 분석 방향을 단순한 "물동량 규모가 얼마인가?"에서 끝낼 필요가 없다는 것을 알 수 있습니다. 전체 물동량이 어떻게 변했는지 볼 수 있고, 그 변화가 수출입과 환적 중 어느 쪽과 더 관련되어 있는지도 살펴볼 수 있습니다.

여기에서 분석 질문을 조금 더 구체화해볼 수 있습니다. 단순히 "부산항 물동량은 얼마인가?"가 아니라 "부산항의 컨테이너 물동량은 이 기간 동안 성장했는가?", "성장했다면 어느 정도 속도로 성장했는가?", "수출입과 환적 중 어느 쪽이 더 빠르게 성장했는가?", "그 결과 환적의 비중은 어떻게 달라졌는가?"와 같은 질문을 던질 수 있게 됩니다.


# 기술통계 확인

In [13]:
# 연도를 제외한 항목 기술통계
origin.iloc[:, 1:].describe()

,컨물동량,증감율,수출입소계,수입,수출,환적소계,환적점유율,환적증감율
count,13.000,13.000,13.000,13.000,13.000,13.000,13.000,13.000
mean,21422.154,2.992,10071.231,5014.000,5057.077,11343.846,52.768,4.395
std,2169.440,2.771,625.807,310.466,317.514,1591.691,2.234,4.460
min,17686.000,-2.770,8934.000,4424.000,4509.000,8748.000,49.470,-4.130
25%,19469.000,1.520,9620.000,4801.000,4819.000,10105.000,50.550,2.100
50%,21824.000,4.040,10233.000,5117.000,5144.000,11638.000,52.920,4.400
75%,22706.000,5.330,10433.000,5208.000,5225.000,12273.000,54.050,7.380
max,24882.000,5.700,10904.000,5409.000,5495.000,14097.000,56.700,11.770


기술통계를 확인해보면, 전체 컨테이너 물동량의 평균은 약 21422이고 중앙값은 21824입니다. 최소값은 17686, 최대값은 24882입니다. 환적소계는 평균 약 11344이고 수출입소계는 평균 약 10071입니다. 환적점유율의 평균도 약 52.77%입니다.

이를 기반으로 첫 번째 분석 가설을 세울 수 있습니다. 관측 기간의 전형적인 컨테이너 물동량은 약 2만 1천 정도이며, 환적 물동량이 수출입 물동량보다 평균적으로 더 큰 규모이고, 전체 물동량 중 절반 이상이 환적이라는 구조가 나타난다는 것입니다.

여기서 중요한 것은 아직 환적이 증가했다라고 판단한 것이 아니라는 부분입니다. 기술통계가 알려주는 평균은 시간 순서를 고려하지 않기 때문입니다. 다만 평균 환적점유율이 50%를 넘고 환적소계의 평균 규모가 수출입소계보다 크므로, '부산항 컨테이너 물동량의 성장 구조를 분석한다면 환적을 별도로 살펴볼 가치가 있겠다'라는 방향성을 잡아볼 수 있습니다.

현재 우리는 물동량의 평균이 얼마인가? 만을 분석하고자 하는 것이 아니라, '13부터 '25까지 물동량이 어떻게 성장했는가를 분석하고자 합니다. 현재 기술통계만으로는 시점이나 시간의 흐름을 확인할 수 없기 때문에, 이제 시간의 흐름을 확인하고 포함하는 과정을 수행해야 합니다.

In [14]:
df = origin.copy()

annual_growth_rate = df['컨물동량'].pct_change() * 100

annual_growth_rate

0       NaN
1     5.637
2     4.207
3    -0.067
4     5.330
5     5.709
6     1.519
7    -0.764
8     4.041
9    -2.766
10    4.874
11    5.390
12    1.967
Name: 컨물동량, dtype: float64

연도별로 정렬되어있는 데이터 기반으로 각 연도의 컨테이너 물동량이 직전 연도에 비해 얼마나 증가하거나 감소했는지를 확인할 수 있습니다.

데이터에 이미 `증감율`이라는 열로 계산이 되어있기는 하지만, 실제로 우리가 분석에 필요한 값이 누락되어있는 상태로 등록되어있는 해(2013)가 있기 때문에 컨물동량의 증감율을 다시 구했습니다.

In [15]:
df.head(1)

,연도,컨물동량,증감율,수출입소계,수입,수출,환적소계,환적점유율,환적증감율
0,2013,17686,3.750,8934,4424,4509,8748,49.470,7.380


실제로 해당 연도의 데이터를 확인해보면, 직전년도(2012)의 물동량 기록 없이 증감율이 3.750이라는 값으로 되어있는 것을 확인할 수 있습니다. 따라서 따라서 2013~2025년 데이터를 이용해 장기 성장률을 구하려면 실제로 필요한 것은 2013 ~ 2014, 2014 ~ 2015, ..., 2024 ~ 2025의 12개 성장 구간입니다.

원본에 있는 증감율을 그대로 사용하면 2012 ~ 2013 구간 하나가 섞이게 됩니다. 따라서 직전의 `annual_growth_rate`처럼 `pct_change()`를 통해ㅐ서 증감율을 직접 구하게 되면 13'의 내용은 NaN(결측치)가 되고, 14' ~ 25' 까지의 전년대비 증감율 데이터가 다시 구해지게 되빈다.

이 상태에서 새로운 질문이 발생합니다. 연도마다 성장률이 다르다면, 2013년부터 2025년까지의 전체적인 장기 성장 속도는 몇 %라고 표현해야 하는 것인가? 해당 질문에 대한 답을 하기 위해서는 성장계수를 적용해야 합니다.

연도별 증감율의 산술평균만으로는 장기적인 성장을 정확히 표현하기 난해합니다. 연도별 증감율을 모두 계산한 다음 평균을 내면 결론적으로는 앞선 결과 위에 다음 변화가 곱해지는 구조로 계산됩니다.

2013년의 물동량에서 2014년의 물동량이 만들어지고, 2014년에 만들어진 규모를 통해서 2015년의 증가율이 적용됩니다. 즉, 연도의 변화는 독립적으로 더해지는 것이 아니라 앞선 결과 위에 다음 변화가 곱해지는 구조인 것입니다. 가령, 100에서 10% 증가하면 110이 되고, 다시 10% 증가하면 이번에는 100의 10%가 아니라 110의 10%가 증가하여 121이 되는 식입니다.

따라서 우리는 증감률을 곱셈이 가능한 형태로 바꿀 수 있어야 합니다. 이 때 사용되는 것이 성장계수입니다.

## 성장계수

성장계수는 어떤 값이 처음보다 몇 배가 되었는지를 나타내는 숫자입니다. 가령, 우리가 은행에 100원을 예금하고 원금이 126원이 된 상황이라고 했을때, 원금대비 26%가 늘었다 라고 이야기하는 것은 '성장률'이 되는 것이고, 1.26배가 되었다 라고 이야기 하는 것은 '성장계수'가 되는 것입니다. 둘은 같은 사실을 다르게 표현한 것뿐이라서, 성장계수는 성장률에 1을 더한 값이 됩니다. 여기서 1은 원래 있던 몫이고 0.26이 새로 늘어난 몫이니까, 원래 있던 것까지 함께 세면 1.26배가 되는 것입니다.

성장계수를 읽을 때는 1을 기준으로 보면 됩니다. 1이면 제자리, 2면 두 배, 0.5면 반토막인 셈입니다. 성장계수가 0.9라면 10% 줄었다는 뜻을 의미합니다. 사라진 것을 세면 0.1이지만 남은 것을 세면 0.9고, 성장계수는 남은 쪽을 읽습니다. 따라서 성장계수는 아무리 줄어들어도 음수가 되지 않고, 최악의 경우가 0입니다.

물동량이 3년 연속 10%씩 늘었다고 할 때 성장률을 그냥 더해서 30%라고 하면 정확하지 않은 결론을 도출해낸 것입니다. 실제로는 33.1%가 늘어난 것이기 때문입니다. 1년차에 10%가 늘어난 값을 기준으로 2년차에 10%가 늘고, 또 2년차에 10%가 늘어난 값을 기준으로 10%가 또 늘어나기 때문입니다.

그런데 성장계수로 바꿔서 1.1을 세 번 곱하면 1.331이 나오고, 이건 정확한 값입니다. 성장률은 더할 수 없지만 성장계수는 곱하면 그대로 이어붙일 수 있기 때문에 성장계수를 사용합니다.

In [16]:
# 증감율을 성장계수로 변환
annual_factor = df["컨물동량"].pct_change().add(1).dropna()

annual_factor

1    1.056
2    1.042
3    0.999
4    1.053
5    1.057
6    1.015
7    0.992
8    1.040
9    0.972
10   1.049
11   1.054
12   1.020
Name: 컨물동량, dtype: float64

In [17]:
# 수치형 데이터를 범주형 데이터로 변환
growth_band = pd.cut(
    annual_growth_rate.dropna(),
    [-float('inf'), 0, 3, 5, float('inf')],
    labels=['감소', '0~3% 미만', '3~5% 미만', '5% 이상']
)

growth_band

1       5% 이상
2     3~5% 미만
3          감소
4       5% 이상
5       5% 이상
6     0~3% 미만
7          감소
8     3~5% 미만
9          감소
10    3~5% 미만
11      5% 이상
12    0~3% 미만
Name: 컨물동량, dtype: category
Categories (4, str): ['감소' < '0~3% 미만' < '3~5% 미만' < '5% 이상']

우리는 지금 부산항의 컨물동량이 2013년부터 2025년까지 어떻게 변했는지를 확인하고 있습니다. pct_change()를 사용하면 각 연도의 전년 대비 증감률을 얻을 수 있습니다. 그런데 여기서 하나의 문제가 생깁니다. 연도마다 증감률이 서로 다릅니다. 어떤 해에는 5% 이상 증가하고, 어떤 해에는 감소하기도 합니다. 실제 데이터에서도 공표 증감율의 최소값은 -2.77%, 최대값은 5.70%로 차이가 있습니다.

그렇다면 이 부분에서 이러한 질문이 가능합니다. 2014년에는 이만큼 성장했고, 2015년에는 또 다르게 성장했고, 중간에는 감소하기도 했는데, 그렇다면 2013년부터 2025년까지 전체적으로 매년 평균 몇 % 정도 성장했다고 표현할 수 있는가?

이러한 상황에서는 이전부터 계속 언급했던 상황(증감'율')때문에 산술평균(우리가 알고 있는 전부 더해서 나누는 그 평균)을 사용할 수 없습니다. 물동량의 성장은 산술적으로 누적되지 않습니다. 매년 기준이 되는 물동량 자체가 달라지기 때문입니다. 그래서 앞에서 증감률을 성장계수로 바꾼 것입니다.

5% 증가라면 성장계수는 1.05, 2% 감소라면 0.98이 됩니다. 이제 각각의 연도 변화는 서로 곱할 수 있는 형태가 됩니다.

매년 실제 성장률은 달랐지만, 만약 12년 동안 매년 똑같은 비율로 성장했다고 가정한다면 그 비율은 얼마여야 처음 값에서 마지막 값에 정확히 도달할 수 있는지에 대한 방법으로 기하평균을 사용할 수 있습니다. 기하평균은 여러 기간 동안 곱셈으로 누적된 서로 다른 성장계수들을 동일한 하나의 성장계수로 바꿔 주는 평균으로 이해하면 됩니다.

정리하면 처음에는 "연도별 물동량이 어떻게 달라졌는가?"를 알고 싶어서 전년 대비 증감률을 구합니다. 그런데 연도마다 증감률이 서로 다르기 때문에 그렇다면 전체 기간의 평균적인 성장 속도는 얼마인가?"라는 새로운 질문이 생깁니다. 그런데 물동량 성장은 매년 직전 연도의 결과를 기준으로 다시 변화하므로 덧셈이 아니라 곱셈으로 누적됩니다. 그래서 증감률을 성장계수로 변환합니다. 성장계수를 여러 해에 걸쳐 곱해서 만들어진 결과를 하나의 동일한 연간 성장계수로 바꾸려면 곱셈 구조를 보존하는 평균이 필요합니다. 그래서 산술평균이 아니라 기하평균이 등장합니다. 그리고 그 기하평균 성장계수에서 1을 빼면 전체 기간의 장기적인 평균 복리 성장률이 됩니다.

In [18]:
start_year = int(df["연도"].iloc[0]) # 관측 시작년도
end_year = int(df["연도"].iloc[-1]) # 마지막년도

# 실제 관측년도
year_span = end_year - start_year
year_span

12

In [19]:
import math

def geometric_mean(s: pd.Series):
    values = pd.to_numeric(s, errors='coerce').dropna()

    if values.empty:
        return float('nan')
    if (values <= 0).any():
        raise ValueError("기하평균은 모든 유효값이 0보다 커야 합니다.")

    # 성장계수를 직접 곱하면 값이 커지거나 0에 가까워지며 부동소수점 정밀도가 손실될 수 있어, 곱셈을 덧셈으로 바꾸는 로그를 취한 뒤 다시 지수로 되돌립니다.
    return math.exp(values.map(math.log).mean())


- geometric_mean 함수의 결론

$$\sqrt[n]{x_1 \times x_2 \times \dots \times x_n}$$

In [20]:
geometric_mean_result = geometric_mean(annual_factor) - 1
endpoints = (
    (df["컨물동량"].iloc[-1] / df["컨물동량"].iloc[0]) ** (1 / year_span) - 1
)

In [21]:
summary = pd.Series({
    "관측 연도 수": len(df),
    "연간 성장 구간 수": len(annual_factor),
    "컨물동량 산술평균": df["컨물동량"].mean(),
    "컨물동량 중앙값": df["컨물동량"].median(),
    "컨물동량 표준편차": df["컨물동량"].std(),
    "환적점유율 평균(%)": df["환적점유율"].mean(),
    "환적점유율 중앙값(%)": df["환적점유율"].median(),
    "가장 흔한 성장 구간": growth_band.mode().iloc[0],
    "기하평균 기반 연평균 성장률(%)": geometric_mean_result * 100,
    "시작·종료값 기반 연평균 성장률(%)": endpoints * 100,
})

summary

관측 연도 수                       13
연간 성장 구간 수                    12
컨물동량 산술평균              21422.154
컨물동량 중앙값               21824.000
컨물동량 표준편차               2169.440
환적점유율 평균(%)               52.768
환적점유율 중앙값(%)              52.920
가장 흔한 성장 구간                5% 이상
기하평균 기반 연평균 성장률(%)         2.886
시작·종료값 기반 연평균 성장률(%)       2.886
dtype: object

컨테이너 물동량의 산술평균은 관측 기간의 대표적인 규모를 보여주고, 중앙값은 연도를 시간순으로 놓았을 때의 가운데 연도 값이 아니라 물동량 크기를 정렬했을 때의 중앙 수준을 의미합니다. 두 값이 크게 다르지 않더라도 이 자료에는 장기 상승 추세가 있으므로 평균과 중앙값만으로 최근 수준이나 성장 방향까지 설명할 수는 없습니다.

증감률은 정확히 같은 값이 반복되는 경우가 적어 원래 값의 최빈값보다 구간별 빈도가 해석하기 용이합니다. 이때 2013년 공표 증감률은 2012년 대비 값이므로 현재 파일 내부의 성장 상태를 셀 때 제외하고, `pct_change()`로 계산되는 2014~2025년의 12개 증감률을 사용했습니다.

연간 성장계수의 기하평균에서 1을 뺀 값과 2013년 ~ 2025년 물동량을 직접 연결한 연평균 성장률은 모두 약 2.886%로 일치합니다. 이는 13개 연도 관측값 사이에 12개의 복리 성장 구간이 있기 때문이었습니다. 산술평균 성장률은 '각 연도의 증감률이 평균적으로 얼마였는가'에 가깝고, 연평균 성장률은 '매년 같은 비율로 성장했다고 가정할 때 시작값에서 종료값에 도달하게 하는 연간 성장률은 얼마인가'에 가깝습니다. 장기 복리 성장 속도를 한 숫자로 비교하려는 목적에는 연평균 성장률이 더 적절하지만, 특정 연도의 급감이나 최근 둔화는 별도의 연도별 증감률과 함께 확인해야 합니다.

# 데이터 정제

In [22]:
df["계산총계"] = df["수출입소계"] + df["환적소계"]
df["계산수출입"] = df["수입"] + df["수출"]
df["계산증감율"] = df["컨물동량"].pct_change() * 100
df["계산성장계수"] = df["컨물동량"].pct_change().add(1)
df["계산환적점유율"] = df["환적소계"] / df["컨물동량"] * 100

df.head()

,연도,컨물동량,증감율,수출입소계,수입,수출,환적소계,환적점유율,환적증감율,계산총계,계산수출입,계산증감율,계산성장계수,계산환적점유율
0,2013,17686,3.750,8934,4424,4509,8748,49.470,7.380,17682,8933,NaN,NaN,49.463
1,2014,18683,5.640,9254,4596,4658,9429,50.470,7.780,18683,9254,5.637,1.056,50.468
2,2015,19469,4.200,9364,4714,4650,10105,51.910,7.170,19469,9364,4.207,1.042,51.903
3,2016,19456,-0.060,9620,4801,4819,9836,50.550,-2.670,19456,9620,-0.067,0.999,50.555
4,2017,20493,5.330,10186,5042,5144,10225,49.900,3.960,20411,10186,5.330,1.053,49.895


# 데이터 품질검사

In [23]:
total_error = df["컨물동량"] - df["계산총계"]
trade_error = df["수출입소계"] - df["계산수출입"]
rate_error = df["증감율"] - df["계산증감율"]
share_error = df["환적점유율"] - df["계산환적점유율"]

max_total_error_idx = total_error.abs().idxmax()

quality = pd.Series({
    "총계-구성합 최대 절대 오차": total_error.abs().max(),
    "총계-구성합 최대 오차 연도": int(df.loc[max_total_error_idx, "연도"]),
    "수출입소계-수입수출합 최대 절대 오차": trade_error.abs().max(),
    "공표-계산 증감률 최대 차이(%p, 2014~2025)": rate_error.iloc[1:].abs().max(),
    "공표-계산 환적점유율 최대 차이(%p)": share_error.abs().max(),
    "완전 중복 행": df.duplicated().sum(),
})

quality

총계-구성합 최대 절대 오차                    82.000
총계-구성합 최대 오차 연도                  2017.000
수출입소계-수입수출합 최대 절대 오차                1.000
공표-계산 증감률 최대 차이(%p, 2014~2025)      0.033
공표-계산 환적점유율 최대 차이(%p)               0.045
완전 중복 행                             0.000
dtype: float64

내용을 확인하여보니 데이터상 계산결과에 오차가 있는것을 확인할 수 있었습니다.

따라서 총계, 수출입소계, 증감률, 환적점유율의 계산 차이는 어느 정도이며, 현재 파일만으로 설명할 수 없는 이상값이 있는가를 확인해보겠습니다.

In [24]:
audit = df[[
    "연도", "컨물동량", "계산총계", "수출입소계", "계산수출입",
    "증감율", "계산증감율", "환적점유율", "계산환적점유율",
]].copy()

audit["총계오차"] = audit["컨물동량"] - audit["계산총계"]
audit["수출입소계오차"] = audit["수출입소계"] - audit["계산수출입"]
audit["증감률오차(%p)"] = audit["증감율"] - audit["계산증감율"]
audit["점유율오차(%p)"] = audit["환적점유율"] - audit["계산환적점유율"]

audit


,연도,컨물동량,계산총계,수출입소계,계산수출입,증감율,계산증감율,환적점유율,계산환적점유율,총계오차,수출입소계오차,증감률오차(%p),점유율오차(%p)
0,2013,17686,17682,8934,8933,3.750,NaN,49.470,49.463,4,1,NaN,0.007
1,2014,18683,18683,9254,9254,5.640,5.637,50.470,50.468,0,0,0.003,0.002
2,2015,19469,19469,9364,9364,4.200,4.207,51.910,51.903,0,0,-0.007,0.007
3,2016,19456,19456,9620,9620,-0.060,-0.067,50.550,50.555,0,0,0.007,-0.005
4,2017,20493,20411,10186,10186,5.330,5.330,49.900,49.895,82,0,0.000,0.005
5,2018,21663,21662,10233,10233,5.700,5.709,52.760,52.758,1,0,-0.009,0.002
6,2019,21992,21992,10354,10354,1.520,1.519,52.920,52.919,0,0,0.001,0.001
7,2020,21824,21824,9804,9804,-0.760,-0.764,55.080,55.077,0,0,0.004,0.003
8,2021,22706,22706,10433,10433,4.040,4.041,54.050,54.052,0,0,-0.001,-0.002
9,2022,22078,22077,10311,10311,-2.770,-2.766,53.290,53.293,1,0,-0.004,-0.003


수출입소계는 수입과 수출의 합과 최대 1만큼 차이가 나고, 2014~2025년의 공표 증감률과 직접 계산한 증감률의 최대 차이는 약 0.033% 정도 입니다. 환적점유율도 직접 계산값과 최대 약 0.045%p 차이여서 이 두 비율의 작은 차이는 공표 과정의 소수점 반올림으로 설명할 수 있는 수준입니다. 2013년 증감률은 비교 기준인 2012년 물동량이 파일에 없으므로 재계산 대상에서 제외합니다.

반면 `컨물동량`과 `수출입소계 + 환적소계`의 차이는 대부분 작지만 2017년에 82인 것을 확인할 수 있스빈다. 따라서 이후 분석에서는 총물동량의 장기 성장률은 공표 `컨물동량`을 기준으로 계산하고, 수출입과 환적은 각각의 공표 소계를 사용하도록 하겠습니다. 그리고 두 구성요소의 증분을 총증분과 비교할 때는 합계 불일치가 분석에 섞이지 않도록 증분잔차를 별도로 계산하도록 하겠습니다.

## 증분

지금까지 우리가 다룬 값은 각 연도의 물동량 자체, 즉 어느 시점의 수준이었습니다. 증분은 그 수준이 아니라 한 시점에서 다음 시점으로 넘어가면서 새로 더해진 양입니다. 2023년 물동량이 23154이고 2024년 물동량이 24402라면 24402는 수준이고 1248이 증분입니다. 그래서 증분은 언제나 직전 시점과 짝을 이루어야만 의미가 생기고, 비교 대상이 없는 첫 해에는 아예 정의되지 않습니다. 우리 자료에 13개 연도 관측값이 있지만 증분은 12개뿐인 이유가 여기에 있고, `diff()`가 첫 행을 결측치로 남기는 것도 같은 이유입니다.

앞에서 우리는 같은 변화를 증감률로도 표현했습니다. 증감률은 직전 값 대비 몇 퍼센트가 달라졌는지를 나타내는 비율이고, 증분은 몇 천 TEU가 달라졌는지를 나타내는 절대량입니다. 두 표현 중 어느 쪽을 쓸지는 그 값으로 무엇을 할 것인지가 결정합니다. 장기 성장 속도를 하나의 숫자로 요약할 때는 매년의 변화가 곱셈으로 누적되므로 비율과 성장계수가 필요했습니다. 반대로 지금부터 하려는 일은 전체의 증가를 수출입과 환적이 각각 얼마씩 나누어 가졌는지 쪼개는 것입니다. 몫을 나누어 다시 더하려면 더할 수 있는 단위여야 하는데, 증감률은 더해지지 않고 절대량인 증분은 더해집니다. 곱셈에는 성장계수가 필요했듯 덧셈에는 증분이 필요한 것입니다.

## 총증분과 구성증분

총증분은 총물동량, 즉 `컨물동량` 열의 증분입니다. 여기에서 '총'은 여러 해에 걸쳐 누적했다는 뜻이 아니라 구성요소가 아닌 전체 항목이라는 뜻이므로, 총증분 역시 다른 증분과 마찬가지로 한 해 동안의 변화량입니다. 이와 짝을 이루는 것이 구성요소의 증분입니다. 수출입증분은 `수출입소계`의 증분이고 환적증분은 `환적소계`의 증분이며, 이 둘을 더한 값을 구성증분합이라고 부르겠습니다.

만약 모든 연도에서 총물동량이 수출입소계와 환적소계의 합과 정확히 일치한다면, 총증분과 구성증분합은 언제나 같아야 합니다. 같은 등식의 양변을 각각 차분한 것에 지나지 않기 때문입니다. 그렇다면 총증분을 수출입증분과 환적증분으로 남김없이 나누어 설명할 수 있고, 우리가 하려는 분해도 그것으로 끝납니다.

## 증분잔차

그런데 앞선 품질검사에서 확인했듯이 이 자료에서는 총계와 구성합이 모든 해에 일치하지 않았습니다. 2017년에는 그 차이가 82였습니다. 그래서 총증분에서 구성증분합을 빼면 0이 아닌 값이 남는 해가 생기고, 이 남는 값을 증분잔차라고 일컫겠습니다.

증분잔차가 어디에서 오는지는 식을 조금만 정리하면 드러납니다. 어느 해의 총계오차를 그해의 총물동량에서 구성합을 뺀 값이라고 하면, 그해의 증분잔차는 그해의 총계오차에서 직전 해의 총계오차를 뺀 값과 정확히 같습니다. 총증분과 구성증분합이 모두 같은 두 해의 차이로 만들어지기 때문입니다. 다시 말해 증분잔차는 새로 생겨난 오차가 아니라, 이미 확인해 둔 총계와 구성합의 불일치가 두 해 사이에 얼마나 달라졌는지를 보여 주는 값입니다. 2018년의 증분잔차가 −81인 것도 2017년의 총계오차 82가 2018년에 1로 줄어들면서 생긴 결과입니다.

증분잔차를 굳이 별도의 열로 두는 이유는 이것이 부산항에서 실제로 일어난 물동량의 변화가 아니라 자료의 성질이기 때문입니다. 이 값을 따로 떼어 놓지 않으면 총증분과 구성증분합의 차이가 수출입이나 환적 쪽으로 흡수되어, 통계 작성 과정의 불일치가 마치 화물 구성의 변화인 것처럼 해석될 수 있습니다. 잔차를 눈에 보이게 남겨 두면 어느 해의 해석이 자료 문제의 영향을 받고 있는지를 그때그때 확인할 수 있습니다.

이제 총증분을 분모로 삼고 수출입증분, 환적증분, 증분잔차를 각각 나누면 그해의 증가를 누가 얼마나 설명하는지를 비율로 표현할 수 있습니다. 다만 이 세 비율의 합은 항상 100%가 되는데, 이는 세 값을 더하면 정의상 총증분이 되기 때문이지 계산이 맞았다는 증거가 아닙니다. 합계가 100%라는 사실은 검산으로 쓸 수 없고, 분모인 총증분이 0에 가깝거나 음수인 해에는 비율 자체가 불안정해진다는 점도 함께 기억해 두어야 합니다.

# 데이터 분석

- 2013 ~ 2025년 13개 연도 관측값을 기준으로 총물동량, 수출입, 환적, 환적점유율의 장기 방향은 무엇이며, 12년간의 연평균 성장률은 얼마인가


In [25]:
period = df[[
    "연도", "컨물동량", "수출입소계", "환적소계", "증감율", "환적점유율"
]].copy()

period

,연도,컨물동량,수출입소계,환적소계,증감율,환적점유율
0,2013,17686,8934,8748,3.750,49.470
1,2014,18683,9254,9429,5.640,50.470
2,2015,19469,9364,10105,4.200,51.910
3,2016,19456,9620,9836,-0.060,50.550
4,2017,20493,10186,10225,5.330,49.900
5,2018,21663,10233,11429,5.700,52.760
6,2019,21992,10354,11638,1.520,52.920
7,2020,21824,9804,12020,-0.760,55.080
8,2021,22706,10433,12273,4.040,54.050
9,2022,22078,10311,11766,-2.770,53.290


In [26]:
def cagr_from_endpoints(series, periods):
    start = series.iloc[0]
    end = series.iloc[-1]

    return (end / start) ** (1 / periods) - 1

In [27]:
year_span = int(df["연도"].iloc[-1] - df["연도"].iloc[0])

total = cagr_from_endpoints(df["컨물동량"], year_span)
trade = cagr_from_endpoints(df["수출입소계"], year_span)
trans = cagr_from_endpoints(df["환적소계"], year_span)

result = pd.Series({
    "총물동량 연평균 성장률": total * 100,
    "수출입 연평균 성장률": trade * 100,
    "환적 연평균 성장률": trans * 100,
    "계산 기간": year_span,
}, name='결과')

result

총물동량 연평균 성장률    2.886
수출입 연평균 성장률     1.581
환적 연평균 성장률      4.056
계산 기간          12.000
Name: 결과, dtype: float64

총물동량은 2013년 17686에서 2025년 24882로 증가했고 환적점유율은 49.47%에서 56.70%로 높아졌습니다. 2013년과 2025년 사이의 12년 연평균 성장률은 총물동량 약 2.886%, 수출입 약 1.581%, 환적 약 4.056%입니다. 환적의 장기 성장 속도가 수출입보다 높았기 때문에 전체 물동량이 증가하는 동안 구성도 환적 쪽으로 이동했다고 해석해 볼 수 있습니다.

다만 연평균 성장률은 시작값과 종료값을 일정한 복리 성장률로 연결한 요약값이므로 중간 연도의 충격을 보여주지는 않습니다. 실제로 2016, 2020, 2022년에는 총물동량이 감소했고, 2020년에는 수출입이 감소하는 동안 환적은 증가했습니다. 따라서 장기적인 연평균 성장률과 연도별 증감률을 함께 봐야 합니다.


### 연간 증감에서 환적이 차지한 몫


In [28]:
contribution = df[["연도", "컨물동량", "수출입소계", "환적소계"]].copy()

contribution["총증분"] = contribution["컨물동량"].diff()
contribution["수출입증분"] = contribution["수출입소계"].diff()
contribution["환적증분"] = contribution["환적소계"].diff()
contribution["구성증분합"] = contribution["수출입증분"] + contribution["환적증분"]
contribution["증분잔차"] = contribution["총증분"] - contribution["구성증분합"]

# 총증분이 0이면 기여율의 분모가 0이 되므로 NaN으로 둡니다.
denominator = contribution["총증분"].where(contribution["총증분"].ne(0))
contribution["수출입기여율"] = contribution["수출입증분"] / denominator * 100
contribution["환적기여율"] = contribution["환적증분"] / denominator * 100
contribution["잔차기여율"] = contribution["증분잔차"] / denominator * 100

contribution["기여율합계"] = (
    contribution["수출입기여율"]
    + contribution["환적기여율"]
    + contribution["잔차기여율"]
)

# 2013년 제외(12년 데이터가 ㅇ벗기 때문에)
contribution[1:]


,연도,컨물동량,수출입소계,환적소계,총증분,수출입증분,환적증분,구성증분합,증분잔차,수출입기여율,환적기여율,잔차기여율,기여율합계
1,2014,18683,9254,9429,997.000,320.000,681.000,1001.000,-4.000,32.096,68.305,-0.401,100.000
2,2015,19469,9364,10105,786.000,110.000,676.000,786.000,0.000,13.995,86.005,0.000,100.000
3,2016,19456,9620,9836,-13.000,256.000,-269.000,-13.000,0.000,-1969.231,2069.231,-0.000,100.000
4,2017,20493,10186,10225,1037.000,566.000,389.000,955.000,82.000,54.581,37.512,7.907,100.000
5,2018,21663,10233,11429,1170.000,47.000,1204.000,1251.000,-81.000,4.017,102.906,-6.923,100.000
6,2019,21992,10354,11638,329.000,121.000,209.000,330.000,-1.000,36.778,63.526,-0.304,100.000
7,2020,21824,9804,12020,-168.000,-550.000,382.000,-168.000,0.000,327.381,-227.381,-0.000,100.000
8,2021,22706,10433,12273,882.000,629.000,253.000,882.000,0.000,71.315,28.685,0.000,100.000
9,2022,22078,10311,11766,-628.000,-122.000,-507.000,-629.000,1.000,19.427,80.732,-0.159,100.000
10,2023,23154,10744,12408,1076.000,433.000,642.000,1075.000,1.000,40.242,59.665,0.093,100.000


2024년 총물동량은 전년보다 1,248 증가했고 수출입은 160, 환적은 1,088 증가했습니다. 당해년도에는 증분잔차가 0이므로 환적증분은 총증분의 약 87.2%를 설명할 수 있습니다.

환적기여율이 100%를 넘는 경우는 원인을 구분해서 봐야 합니다. 2025년에는 총증분이 480인데 환적은 601 증가하고 수출입은 119 감소했으며 증분잔차는 -2입니다. 따라서 환적 증가가 수출입 감소를 주로 상쇄하면서 환적기여율이 약 125.2%가 됩니다. 반면 2018년에는 수출입도 47 증가했는데 환적증분이 1,204로 총증분 1,170보다 큽니다. 이는 2017년에 존재하던 큰 총계-구성합 차이가 2018년에 크게 줄면서 증분잔차가 -81이 되었기 때문으로 볼 수 있습니다. 따라서 2018년의 100% 초과를 단순히 수출입 감소 때문이라고 설명할 수는 없습니다.

또한 총증분이 -13에 불과한 2016년처럼 분모의 절대값이 매우 작으면 기여율이 수천 %까지 커질 수 있다. 총증분이 음수인 해에는 기여율의 부호도 직관적이지 않을 수 있습니다. 그러므로 기여율만 순위화하지 말고 총증분, 수출입증분, 환적증분, 증분잔차의 부호와 절대량을 먼저 확인한 뒤 보조적으로 비율을 사용해야 합니다.

# 결론

해당 자료는 2013 ~ 2025년의 13개 연도 관측값을 담고 있으며, 그 사이에는 12개의 연간 성장 구간이 있습니다. 2014 ~ 2025년의 공표 증감률과 직접 계산한 증감률, 그리고 공표 환적점유율과 직접 계산한 점유율은 작은 차이만 확인됩니다. 반면 공표 총물동량과 수출입소계 + 환적소계는 2017년에 82 차이가 확인되었습니다.

장기 성장에서는 2013년 17686이 2025년 24882로 증가했고, 12년 연평균 성장률은 총물동량 약 2.886%, 수출입 약 1.581%, 환적 약 4.056% 정도가 됩니다. 환적점유율도 49.47% 에서 56.70% 로 높아져 관측 기간 동안 환적의 상대적 비중이 커졌음을 확인할 수 있습니다, 연평균 성장률은 장기 속도를 한 숫자로 비교하는 데 적합하지만 중간 연도의 감소나 최근 둔화를 숨길 수 있으므로 연도별 증감률과 함께 읽어야 합니다.

증감량 분해에서는 공표 총계와 구성합이 완전히 일치하지 않는 문제를 반영해 증분잔차를 추가했습니다. 2024년에는 총증분 1,248 가운데 환적증분이 1,088로 약 87.2%를 설명하고 잔차는 0입니다. 2025년에는 환적이 601 증가하고 수출입이 119 감소했으며 잔차 -2까지 포함해 총증분 480이 됩니다. 2018년의 환적기여율 100% 초과는 수출입 감소가 아니라 전년과 당해 연도의 총계-구성합 차이가 달라져 생긴 잔차의 영향이므로 별도로 해석해볼 수 있습니다.

In [ ]:
import mysql.connector